# 09 · Poverty vulnerability model

**Question.** How well can EICV7 household characteristics, the CPI shock and the distress score rank households by their risk of being poor? Do the models beat a simple district baseline?

| | |
|---|---|
| **Target** | `poor` (real consumption per adult equivalent below the poverty line) |
| **Features** | The `full` set (119): geography, demographics, education, program-free employment, private income, ICT and finance, assets, housing, services, expenditure shares, CPI shock and `distress_score_model`. Sensitivity: the `pmt` set, without consumption-derived features. |
| **Models** | District poverty-rate baseline, logistic regression, random forest, LightGBM, XGBoost. Library defaults: **no tuning**. |
| **Validation** | 5-fold CV over whole clusters within strata (development set); final holdout scored once |
| **Code** | `src/models/poverty_model.py`, `src/models/common.py` |
| **Outputs** | `reports/model_results_poverty.csv`, `models/poverty_vulnerability_pipeline.joblib` (+ `.json` metadata), `data/processed/poverty_scores.parquet`, `reports/figures/poverty_model/*.png` |

**Weights.**
- Training uses the household weight, normalised to mean 1.
- Household metrics use `weight`.
- "Recall (people)" is the share of poor **people** reached, using `pop_wt`.

**Top-k metrics.** The top *k*% of the weighted household population is selected by predicted risk. Ties at the cut-off are included fractionally, so the district baseline is not favoured or penalised by arbitrary tie-breaking.

**Selection rule (fixed in advance).**
1. Take the model with the highest mean fold PR-AUC.
2. Among the models within one standard error of it, choose the simplest.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "models" / "common.py").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from sklearn.metrics import precision_recall_curve, roc_curve

from src.models import common as cm
from src.models import plots as pl
from src.models import poverty_model as pm

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)
pm.FIG_DIR.mkdir(parents=True, exist_ok=True)

df = cm.load_frame().merge(cm.load_splits(), on=["hhid", "clust"])
dev = df[df["split"].eq("development")].reset_index(drop=True)
hold = df[df["split"].eq("holdout")].reset_index(drop=True)
spec = pm.FEATURE_SETS["full"]
cm.assert_no_leakage(spec, "poverty")
cm.assert_no_leakage(pm.FEATURE_SETS["pmt"], "poverty")
print(f"Development: {len(dev):,} households, {dev['clust'].nunique()} clusters; holdout: {len(hold):,} households, {hold['clust'].nunique()} clusters")
print(f"Poverty rate (households, weight): development {np.average(dev['poor'], weights=dev['weight']):.3f}; holdout {np.average(hold['poor'], weights=hold['weight']):.3f}")
print(f"Features: full {len(spec.columns)}, pmt {len(pm.FEATURE_SETS['pmt'].columns)}; categorical: {spec.categorical}")

## 1. Cross-validation on the development set

In [ ]:
t0 = time.time()
pipes, cols = pm.make_pipelines(spec)
oof = cm.cross_validate(pipes, dev, cols, pm.TARGET, dev["fold"])
fm = pm.fold_metrics(oof, dev, dev["fold"])
print(f"CV done in {time.time() - t0:.0f}s")
show = ["roc_auc", "pr_auc", "brier", "precision_top10", "recall_people_top10", "precision_top20", "recall_people_top20",
        "precision_top30", "recall_people_top30", "calibration_slope", "calibration_in_the_large"]
cv_summary = fm.groupby("model")[show].agg(["mean", "std"])
cv_summary.loc[pm.MODELS]

In [ ]:
oof_metrics = pd.DataFrame([{"model": n, **cm.evaluate(dev[pm.TARGET], oof[n], dev["weight"], dev["pop_wt"])} for n in pm.MODELS])
display(oof_metrics.set_index("model")[show])
best, sel = pm.select_best(fm)
display(sel)
print(f"Selected model (rule: best mean fold {pm.SELECTION_METRIC}, simplest within 1 SE): {best}")

In [ ]:
# Fold-paired comparison with the district baseline: every fold, every model
paired = fm.pivot_table(index="fold", columns="model", values=["roc_auc", "pr_auc", "recall_people_top20"])
gain = pd.DataFrame({m: {metric: (paired[metric][m] - paired[metric]["district_baseline"]).agg(["mean", "min"]).round(3).to_dict()
                         for metric in ["roc_auc", "pr_auc", "recall_people_top20"]} for m in pm.COMPLEXITY}).T
print("Gain over the district baseline across the 5 folds (mean and worst fold):")
gain

In [ ]:
fig, axes = pl.figure("Every model beats the district baseline in every fold",
                      "Five cluster-based cross-validation folds (development set); each dot is one fold", width=11, height=4.3, ncols=3)
for ax, metric, label in zip(axes, ["roc_auc", "pr_auc", "recall_people_top20"], ["ROC-AUC", "PR-AUC", "Poor people reached, top 20%"]):
    for i, m in enumerate(pm.MODELS[::-1]):
        v = fm.loc[fm["model"].eq(m), metric]
        ax.plot(v, [i] * len(v), "o", color=pl.MODEL_COLORS[m], alpha=0.75, markersize=6)
        ax.plot([v.mean()], [i], "|", color=pl.INK["primary"], markersize=14, markeredgewidth=2)
    ax.set_yticks(range(len(pm.MODELS)), [pl.MODEL_LABELS[m] for m in pm.MODELS[::-1]] if metric == "roc_auc" else [""] * len(pm.MODELS), fontsize=9)
    ax.set_title(label, fontsize=10, color=pl.INK["secondary"], loc="left")
path = pl.save(fig, pm.FIG_DIR / "cv_fold_metrics.png", top=0.74, bottom=0.14, left=0.15, wspace=0.12)
display(Image(str(path)))

## 2. Final holdout

Every model is refitted on the full development set and scored once on the untouched holdout. Intervals come from a cluster bootstrap (500 draws; clusters resampled within strata). "Difference vs baseline" intervals are paired: the same draws are used for both models.

In [ ]:
t0 = time.time()
fitted = pm.fit_on(pipes, cols, dev)
hold_pred = pm.predict(fitted, cols, hold)
hold_metrics = pd.DataFrame([{"model": n, **cm.evaluate(hold[pm.TARGET], hold_pred[n], hold["weight"], hold["pop_wt"])} for n in pm.MODELS])
hb = hold.join(hold_pred)
boot = cm.cluster_bootstrap(hb, pm.MODELS, pm.TARGET, B=500, baseline="district_baseline",
                            metrics=("roc_auc", "pr_auc", "brier", "precision_top10", "recall_people_top10", "precision_top20",
                                     "recall_people_top20", "precision_top30", "recall_people_top30"))
print(f"Holdout scoring and bootstrap done in {time.time() - t0:.0f}s")
display(hold_metrics.set_index("model")[show])

In [ ]:
ci = boot.assign(interval=lambda b: b.apply(lambda r: f"{r.ci_low:.3f}-{r.ci_high:.3f}", axis=1),
                 diff=lambda b: b.apply(lambda r: "" if pd.isna(r.get("diff_vs_baseline_ci_low")) else f"{r.diff_vs_baseline_ci_low:+.3f} to {r.diff_vs_baseline_ci_high:+.3f}", axis=1))
display(ci.pivot_table(index="score", columns="metric", values="interval", aggfunc="first").loc[pm.MODELS, ["roc_auc", "pr_auc", "recall_people_top20", "precision_top20"]])
print("Paired difference vs district baseline, 95% interval:")
display(ci[ci["score"].ne("district_baseline")].pivot_table(index="score", columns="metric", values="diff", aggfunc="first")[["roc_auc", "pr_auc", "recall_people_top20", "precision_top20", "brier"]])

In [ ]:
fig, (a1, a2) = pl.figure("Holdout ROC and precision-recall curves", "Final holdout (3,022 households), models fitted on the development set; AUC in brackets",
                          width=11, height=5, ncols=2)
y, w = hold[pm.TARGET], hold["weight"]
roc = {}
pr = {}
for n in pm.MODELS:
    fpr, tpr, _ = roc_curve(y, hold_pred[n], sample_weight=w)
    p_, r_, _ = precision_recall_curve(y, hold_pred[n], sample_weight=w)
    e = hold_metrics.set_index("model").loc[n]
    roc[n], pr[n] = (fpr, tpr, e["roc_auc"]), (r_, p_, e["pr_auc"])
pl.curves(a1, "roc", roc)
pl.curves(a2, "pr", pr, baseline_rate=np.average(y, weights=w))
path = pl.save(fig, pm.FIG_DIR / "holdout_roc_pr.png", top=0.82, bottom=0.12, left=0.07, wspace=0.2)
display(Image(str(path)))

In [ ]:
fig, (ax,) = pl.figure("Calibration on the holdout", "Observed poverty rate (households) vs mean predicted probability, by decile of prediction", width=7, height=6)
ax.plot([0, 1], [0, 1], ":", color=pl.BASELINE)
for n in ["district_baseline", "logistic_regression", best] if best != "logistic_regression" else ["district_baseline", "logistic_regression", "lightgbm"]:
    t = cm.calibration_table(y, hold_pred[n], w)
    ax.plot(t["mean_predicted"], t["observed"], "o-", color=pl.MODEL_COLORS[n], linewidth=2, markersize=5, label=pl.MODEL_LABELS[n])
ax.set_xlabel("Mean predicted probability", fontsize=9, color=pl.INK["secondary"])
ax.set_ylabel("Observed share poor", fontsize=9, color=pl.INK["secondary"])
ax.grid(axis="both", color=pl.GRID)
ax.legend(frameon=False, fontsize=8.5)
path = pl.save(fig, pm.FIG_DIR / "holdout_calibration.png", top=0.86, bottom=0.1, left=0.12)
display(Image(str(path)))
display(hold_metrics.set_index("model")[["calibration_slope", "calibration_intercept", "calibration_in_the_large", "brier"]])

## 3. Prioritising limited resources

A budget can reach a fixed share of households. The targeting curve shows the share of **poor people** reached for each share of households selected, ranking by predicted risk. The current VUP allocation is shown as a point at its own coverage.

In [ ]:
vup_alloc = pm.allocation_metrics(hold[pm.TARGET], hold["vup_any"], hold["weight"], hold["pop_wt"])
k_vup = vup_alloc["share_selected"]
fig, (ax,) = pl.figure("Share of poor people reached for each share of households targeted",
                       "Final holdout. Ranking by predicted poverty risk; dot = current VUP allocation at its own coverage", width=9, height=6)
ax.plot([0, 100], [0, 100], ":", color=pl.BASELINE, label="Random selection")
tc = {}
for n in pm.MODELS:
    tc[n] = pm.targeting_curve(hold[pm.TARGET], hold_pred[n], hold["weight"], hold["pop_wt"])
    ax.plot(100 * tc[n]["k"], 100 * tc[n]["recall_people"], "--" if n == "district_baseline" else "-", color=pl.MODEL_COLORS[n], linewidth=2, label=pl.MODEL_LABELS[n])
ax.plot(100 * k_vup, 100 * vup_alloc["recall_people"], "D", color=pl.SERIES[7], markersize=9, markeredgecolor=pl.SURFACE, label="Current VUP")
for k in (10, 20, 30):
    ax.axvline(k, color=pl.GRID, linewidth=1, zorder=0)
ax.set_xlabel("Households targeted, % (ranked by predicted risk)", fontsize=9, color=pl.INK["secondary"])
ax.set_ylabel("Poor people reached, %", fontsize=9, color=pl.INK["secondary"])
ax.grid(axis="both", color=pl.GRID)
ax.legend(frameon=False, fontsize=8.5, loc="lower right")
path = pl.save(fig, pm.FIG_DIR / "holdout_targeting_curve.png", top=0.86, bottom=0.1, left=0.1)
display(Image(str(path)))
same_budget = {n: cm.evaluate(hold[pm.TARGET], hold_pred[n], hold["weight"], hold["pop_wt"], k_values=(k_vup,)) for n in pm.MODELS}
tag = f"top{int(round(100 * k_vup))}"
budget = pd.DataFrame({n: {"precision": e[f"precision_{tag}"], "poor people reached": e[f"recall_people_{tag}"]} for n, e in same_budget.items()}).T
budget.loc["current VUP allocation"] = [vup_alloc["precision"], vup_alloc["recall_people"]]
print(f"At the current VUP budget ({100 * k_vup:.1f}% of households):")
budget

## 4. Subgroup performance

The selected model and the district baseline, scored on the holdout:
- **ROC-AUC within the group:** does the model rank households correctly inside the group?
- **Calibration gap:** mean predicted minus observed poverty rate.
- **Recall at the national top 20%:** the share of the group's poor people that a national top-20% ranking reaches.

Intervals come from a cluster bootstrap (200 draws). Groups with fewer than 20 poor sample households have no AUC.

In [ ]:
hb["income_source"] = hb["main_livelihood_pf"]
sub = {}
for by in ["province_name", "area", "income_source"]:
    for n in [best, "district_baseline"]:
        sub[(by, n)] = cm.subgroup_bootstrap(hb, n, pm.TARGET, by, B=200)
cols_show = ["households", "positives", "observed_rate", "mean_predicted", "calibration_gap", "roc_auc", "roc_auc_ci_low", "roc_auc_ci_high",
             "recall_people_at_national_top20", "recall_people_at_national_top20_ci_low", "recall_people_at_national_top20_ci_high"]
subgroups = pd.concat({k: v[cols_show] for k, v in sub.items()}, names=["dimension", "model", "group"])
subgroups

In [ ]:
LIV = {"own_farming": "own farming", "farm_wage": "farm wage work", "nonfarm_wage": "non-farm wage work",
       "nonfarm_self_employment": "non-farm self-employment", "no_employed_member": "no employed member"}
rows = []
for by, label in [("province_name", "Province"), ("area", "Area"), ("income_source", "Income")]:
    t = sub[(by, best)]
    for g, r in t.iterrows():
        rows.append({"group": f"{label}: {LIV.get(g, str(g).replace(' Province', ''))}", "households": r["households"],
                     "mean": r["roc_auc"], "ci_low": r["roc_auc_ci_low"], "ci_high": r["roc_auc_ci_high"]})
tab = pd.DataFrame(rows).set_index("group").dropna()
overall = hold_metrics.set_index("model").loc[best, "roc_auc"]
fig, (ax,) = pl.figure(f"{pl.MODEL_LABELS[best]}: ranking quality within subgroups (holdout)",
                       "ROC-AUC within each group with 95% cluster-bootstrap interval; dashed line = overall holdout AUC", width=10, height=1.9 + 0.36 * len(tab))
pl.dot_ci(ax, tab, 1, pl.MODEL_COLORS[best], label_fmt="{:.2f}", reference=overall)
ax.set_xlim(min(0.6, tab["ci_low"].min() - 0.02), 1.0)
path = pl.save(fig, pm.FIG_DIR / "holdout_subgroup_auc.png", top=1 - 1.25 / (1.9 + 0.36 * len(tab)), bottom=0.5 / (1.9 + 0.36 * len(tab)), left=0.36)
display(Image(str(path)))

## 5. Sensitivity: PMT-feasible features

The same models without expenditure shares, the CPI shock and the distress score. All three come from the consumption module, so they are not available for households outside the survey. This shows how much of the performance depends on consumption data.

In [ ]:
spec_pmt = pm.FEATURE_SETS["pmt"]
pipes_pmt, cols_pmt = pm.make_pipelines(spec_pmt, models=["district_baseline", "logistic_regression", "lightgbm"])
oof_pmt = cm.cross_validate(pipes_pmt, dev, cols_pmt, pm.TARGET, dev["fold"])
fm_pmt = pm.fold_metrics(oof_pmt, dev, dev["fold"])
fitted_pmt = pm.fit_on(pipes_pmt, cols_pmt, dev)
hold_pmt = pm.predict(fitted_pmt, cols_pmt, hold)
hold_metrics_pmt = pd.DataFrame([{"model": n, **cm.evaluate(hold[pm.TARGET], hold_pmt[n], hold["weight"], hold["pop_wt"])} for n in pipes_pmt])
boot_pmt = cm.cluster_bootstrap(hold.join(hold_pmt), list(pipes_pmt), pm.TARGET, B=500, baseline="district_baseline",
                                metrics=("roc_auc", "pr_auc", "brier", "precision_top10", "recall_people_top10", "precision_top20",
                                         "recall_people_top20", "precision_top30", "recall_people_top30"))
cmp = pd.concat({"full": hold_metrics.set_index("model").loc[["logistic_regression", "lightgbm"]],
                 "pmt": hold_metrics_pmt.set_index("model").loc[["logistic_regression", "lightgbm"]]})[["roc_auc", "pr_auc", "recall_people_top20", "precision_top20"]]
print("Holdout, full vs PMT-feasible features:")
display(cmp)
print("CV fold means, PMT-feasible:")
display(fm_pmt.groupby("model")[["roc_auc", "pr_auc", "recall_people_top20"]].mean())

## 6. Save

In [ ]:
res = pd.concat([pm.results_table(fm, oof_metrics, hold_metrics, boot, "full", best),
                 pm.results_table(fm_pmt, pd.DataFrame([{"model": n, **cm.evaluate(dev[pm.TARGET], oof_pmt[n], dev["weight"], dev["pop_wt"])} for n in pipes_pmt]),
                                  hold_metrics_pmt, boot_pmt, "pmt", "")], ignore_index=True)
res.to_csv(pm.RESULTS_PATH, index=False)
print(f"{pm.RESULTS_PATH.relative_to(ROOT)}: {len(res)} rows")

best_pipe = fitted[best]
meta = {"model": best, "feature_set": "full", "target": pm.TARGET, "feature_columns": cols[best],
        "trained_on": f"development split: {len(dev):,} households, {dev['clust'].nunique()} clusters (data/processed/model_splits.parquet)",
        "sample_weight": "household weight normalised to mean 1", "hyperparameters": "library defaults (no tuning)",
        "selection_rule": f"highest mean fold {pm.SELECTION_METRIC}; simplest within 1 SE",
        "holdout_metrics": hold_metrics.set_index("model").loc[best, ["roc_auc", "pr_auc", "brier", "precision_top20", "recall_people_top20"]].round(4).to_dict(),
        "usage": "joblib.load(path).predict_proba(frame[feature_columns])[:, 1]; build the frame with src.models.common.load_frame()"}
pm.save_pipeline(best_pipe, pm.PIPELINE_PATH, meta)
print(f"Saved {pm.PIPELINE_PATH.relative_to(ROOT)} and metadata {pm.PIPELINE_PATH.with_suffix('.json').relative_to(ROOT)}")

# Poverty vulnerability scores for notebook 10: out-of-fold for development, development-fit model for holdout
scores = pd.concat([dev[["hhid", "split", "fold"]].assign(poverty_score=oof[best].to_numpy()),
                    hold[["hhid", "split", "fold"]].assign(poverty_score=hold_pred[best].to_numpy())], ignore_index=True)
scores.to_parquet(pm.SCORES_PATH, index=False)
print(f"{pm.SCORES_PATH.relative_to(ROOT)}: {len(scores):,} households (model: {best})")